# «Своё Вино» — AR-сканер в реальном времени на Kaggle T4 (работает и с бутылкой воды)

**Settings:** Accelerator → **GPU T4 x2** (хватает одной), Internet → **On**.

**Add Input** (аккаунт `tmama23`):
1. `tmama23/wine-ml-service-shelf` — сервис распознавания: полка, маски `/rt/masks`, тип напитка (вода / газировка / пиво / сок / крепкое) для бутылок не из каталога
2. `tmama23/wine-web-build` — собранный сайт с живым AR (`/scan/ar`, `/scan/ar-track`)
3. `tmama23/wine-track-assets` — дообученная SigLIP2 (vision, fp16)

**Run All.** Через 3–7 минут в выводе появятся ссылки `https://<слова>.trycloudflare.com/scan/ar` — открыть на телефоне (https, камера разрешится).
- Вино из каталога → метка с названием на бутылке + карточка снизу.
- Бутылка воды / газировки / пива → панель «Это не вино · Бутылка воды».
- Вино не из каталога → «Нет в каталоге» и подбор похожих.

Ячейка держит всё 8 часов; остановка ноутбука = ссылка перестаёт работать.

In [ ]:
# wine-site-serve (GPU T4): the whole scanner on Kaggle for phone testing — recognition service (ml/, dataset qujvxta/wine-ml-service,
# tuned SigLIP2 + RF-DETR, shelf mode) on :8000, the Nuxt site (prebuilt .output, dataset qujvxta/wine-web-build) on :3000,
# and a Cloudflare quick tunnel to the SITE. The public URL is posted to ntfy.sh/svoe-vino-7519a1a8ca3b (the log of a running batch kernel is not readable).
import glob, json, os, re, subprocess, sys, tarfile, threading, time, urllib.request
from pathlib import Path
TOPIC = ""   # optional: an ntfy.sh topic to also receive the URL on the phone
HOURS = 8
W = Path("/kaggle/working"); T0 = time.time()
def log(m): print(f"[{time.time()-T0:6.0f}s] {m}", flush=True)
def notify(m):
    if not TOPIC: return
    try: urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{TOPIC}", data=m.encode()), timeout=10)
    except Exception as e: log(f"notify failed {e!r}")
def find(p): h = sorted(glob.glob(p, recursive=True), key=len); return h[0] if h else None

# ---- recognition service (serve.py from the code dataset, no tunnel of its own) ----
serve = find("/kaggle/input/**/kaggle/serve.py")
log(f"serve.py {serve}")
notify("stage: start, launching ml service")
ml_env = dict(os.environ, RT_DET_COMPILE="none", RT_DET_PRECISION="amp")   # no torch.compile max-autotune at start-up (minutes of tuning)
ml = subprocess.Popen([sys.executable, serve, "--no-tunnel", "--selftest", "0", "--hours", str(HOURS), "--port", "8000"],
                      stdout=open(W / "ml.log", "w"), stderr=subprocess.STDOUT, env=ml_env)
# ---- node + site ----
NODE = "v22.11.0"
urllib.request.urlretrieve(f"https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz", "/tmp/node.tar.xz")
subprocess.run(["tar", "-xJf", "/tmp/node.tar.xz", "-C", "/tmp"], check=True)
node = f"/tmp/node-{NODE}-linux-x64/bin/node"
site = Path(find("/kaggle/input/**/server/index.mjs")).parents[1]   # the dataset root holds nitro.json, public/, server/
log(f"site {site}")
for _ in range(1200):   # wait for the ML service (model load + warm-up)
    try: urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2); break
    except Exception:
        if ml.poll() is not None: tail = open(W / "ml.log").read()[-1500:]; notify("ML DIED: " + tail); raise SystemExit("ml died")
        time.sleep(2)
log("ml up"); notify(f"stage: ml up in {time.time()-T0:.0f}s")
env = dict(os.environ, PORT="3000", HOST="0.0.0.0", NUXT_ML_URL="http://127.0.0.1:8000", NUXT_ML_TIMEOUT_MS="20000", NUXT_ML_URL_FILE="/tmp/none.txt")
web = subprocess.Popen([node, str(site / "server" / "index.mjs")], env=env, cwd="/tmp", stdout=open(W / "web.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try: urllib.request.urlopen("http://127.0.0.1:3000/wines", timeout=3); break
    except Exception: time.sleep(1)
log("site up"); notify(f"stage: site up in {time.time()-T0:.0f}s, starting tunnel")
# ---- tunnel to the site ----
cf = W / "cloudflared"
urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf); cf.chmod(0o755)
def tunnel():
    p = subprocess.Popen([str(cf), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:3000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    while url is None:
        line = p.stdout.readline()
        if not line: time.sleep(0.2); continue
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: url = m.group(0)
    threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()
    for _ in range(90):
        try: urllib.request.urlopen(f"{url}/wines", timeout=6); break
        except Exception: time.sleep(2)
    return p, url
tp, url = tunnel()
log(f"PUBLIC SITE {url}/wines"); notify(f"SITE {url}/wines")
print("\n" + "=" * 70 + f"\n  AR В РЕАЛЬНОМ ВРЕМЕНИ:  {url}/scan/ar\n  трекинг этикеток:      {url}/scan/ar-track\n  сайт:                  {url}/wines\n" + "=" * 70 + "\n", flush=True)
try:
    from IPython.display import display, HTML
    display(HTML(f'<h2>AR: <a href="{url}/scan/ar" target="_blank">{url}/scan/ar</a></h2>'
                 f'<p>трекинг этикеток: <a href="{url}/scan/ar-track" target="_blank">{url}/scan/ar-track</a> · '
                 f'сайт: <a href="{url}/wines" target="_blank">{url}/wines</a></p>'))
except Exception: pass
(W / "site_url.txt").write_text(url + "\n")
end = T0 + HOURS * 3600
while time.time() < end:
    time.sleep(30)
    if ml.poll() is not None: notify("ML DIED"); print(open(W / "ml.log").read()[-3000:]); break
    if web.poll() is not None:
        web = subprocess.Popen([node, str(site / "server" / "index.mjs")], env=env, cwd="/tmp", stdout=open(W / "web.log", "a"), stderr=subprocess.STDOUT)
    if tp.poll() is not None:
        tp, url = tunnel(); notify(f"NEW SITE {url}/wines"); print(f"NEW SITE URL: {url}/wines", flush=True)
log("done")
